# 11 · Stage1 A7 보존 + Stage3 v4-A 제출 빌드

이 노트북은 새 Colab 런타임에서 바로 실행할 수 있도록 다음 순서로 시작합니다.

1. Google Drive mount
2. `stage3-sangchun` 브랜치 clone 또는 fetch/pull
3. build script 존재 확인
4. 제출 빌드 실행

`stage1_a7.zip`은 build script가 Drive에서 자동 탐색합니다.


In [ ]:
from __future__ import annotations

import subprocess
from pathlib import Path

from google.colab import drive

# ============================================================
# 1. Google Drive
# ============================================================
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
if not DRIVE_ROOT.exists():
    raise FileNotFoundError(
        f"Drive project root not found: {DRIVE_ROOT}"
    )

# ============================================================
# 2. Repository clone / update
# ============================================================
REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    print(f"Cloning {BRANCH}...")
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO),
        ],
        check=True,
    )
else:
    print("Repository already exists:", REPO)

    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()

    if dirty:
        raise RuntimeError(
            "Colab repository has local changes. "
            "Submission build must start from a clean checkout.\n\n"
            + dirty
            + "\n\nUse a fresh runtime or commit/stash the changes first."
        )

    subprocess.run(
        ["git", "-C", str(REPO), "fetch", "origin", BRANCH],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO), "checkout", BRANCH],
        check=True,
    )
    subprocess.run(
        [
            "git",
            "-C",
            str(REPO),
            "pull",
            "--ff-only",
            "origin",
            BRANCH,
        ],
        check=True,
    )

commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

print("Repository:", REPO)
print("Branch    :", BRANCH)
print("Commit    :", commit)

# ============================================================
# 3. Build script contract
# ============================================================
SCRIPT = (
    REPO
    / "scripts"
    / "stage3"
    / "build_stage1a7_stage3_submission.py"
)

if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"Build script is still missing after git pull: {SCRIPT}\n"
        "Make sure the stage1a7+stage3 patch was committed and pushed "
        f"to origin/{BRANCH}."
    )

print("Build script:", SCRIPT)
print("exists      :", SCRIPT.is_file())


In [ ]:
# ============================================================
# 4. Run reproducible submission build
# ============================================================
import runpy

runpy.run_path(
    str(SCRIPT),
    run_name="__main__",
)
